<a href="https://colab.research.google.com/github/IT25101804/Group_58/blob/main/notebooks/IT25100879LE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder #import le
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              average_precision_score, f1_score, accuracy_score, recall_score,
                              precision_score)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not available -> will use GradientBoostingClassifier instead.")

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

In [ ]:
import os as _os
_FIG_DIR = "eda_visualizations"
_os.makedirs(_FIG_DIR, exist_ok=True)
_FIG_COUNTER = {"n": 0}
_NOTEBOOK_TAG = "member4_IT25100879"
_orig_plt_show = plt.show
def _show_and_save(*args, **kwargs):
    _FIG_COUNTER["n"] += 1
    _fname = _os.path.join(_FIG_DIR, f"{_NOTEBOOK_TAG}_fig{_FIG_COUNTER['n']:02d}.png")
    try:
        plt.savefig(_fname, dpi=120, bbox_inches="tight")
    except Exception as _e:
        print("Could not save figure:", _e)
    return _orig_plt_show(*args, **kwargs)
plt.show = _show_and_save
print("Figure autosave enabled -> saving to", _os.path.abspath(_FIG_DIR))

Figure autosave enabled -> saving to /content/eda_visualizations


In [ ]:
#d load


try:
    from google.colab import files
    print("Running in Google Colab. Please choose 'diabetic_data.csv' AND 'IDS_mapping.csv' to upload:")
    uploaded = files.upload()
    DATA_PATH = "diabetic_data.csv"
    IDS_MAPPING_PATH = "IDS_mapping.csv"
except ImportError:
    DATA_PATH = "diabetic_data.csv"
    IDS_MAPPING_PATH = "IDS_mapping.csv"
    print(f"Not running in Colab -- looking for local files '{DATA_PATH}' and '{IDS_MAPPING_PATH}'.")

#dont no dc
df = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[])
print("Loaded dataset:", df.shape)
df.head()

Running in Google Colab. Please choose 'diabetic_data.csv' AND 'IDS_mapping.csv' to upload:


Saving diabetic_data.csv to diabetic_data.csv
Saving IDS_mapping.csv to IDS_mapping.csv
Loaded dataset: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [ ]:
# Function to load ID mapping tables


import csv

def load_ids_mapping(path):
    tables = {}
    current_key, current_rows = None, []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.reader(f):
            if len(row) < 2:
                continue
            col0, col1 = row[0].strip(), row[1].strip()
            if col1.lower() == "description" and col0.endswith("_id"):
                if current_key:
                    tables[current_key] = dict(current_rows)
                current_key, current_rows = col0, []
                continue
            if col0 == "" and col1 == "":
                continue
            if current_key and col0 != "":
                try:
                    current_rows.append((int(col0), col1))
                except ValueError:
                    pass
        if current_key:
            tables[current_key] = dict(current_rows)
    return tables

ids_mapping = load_ids_mapping(IDS_MAPPING_PATH)
for key, mapping in ids_mapping.items():
    print(f"{key}: {len(mapping)} codes decoded")

admission_type_map = ids_mapping["admission_type_id"]
discharge_disposition_map = ids_mapping["discharge_disposition_id"]
admission_source_map = ids_mapping["admission_source_id"]

print("\nExample decodes:")
print(" discharge_disposition_id 11 ->", discharge_disposition_map[11])
print(" discharge_disposition_id 1  ->", discharge_disposition_map[1])
print(" admission_type_id 1         ->", admission_type_map[1])

admission_type_id: 8 codes decoded
discharge_disposition_id: 30 codes decoded
admission_source_id: 25 codes decoded

Example decodes:
 discharge_disposition_id 11 -> Expired
 discharge_disposition_id 1  -> Discharged to home
 admission_type_id 1         -> Emergency


In [ ]:
# Initial Data Inspection & Quality Check


print("Shape:", df.shape)
print("\n")
df.info()
print("\n")
print(df.describe())

# miss
missing_q = (df == "?").sum()
print("\nColumns using '?' as a disguised missing-value placeholder:")
print(missing_q[missing_q > 0])

print("\n...as a percentage of all rows:")
print((missing_q[missing_q > 0] / len(df) * 100).round(2))

print("\nExact duplicate rows:", df.duplicated().sum())

# dead
expired_count = (df['discharge_disposition_id'].astype(int).isin([11, 19, 20, 21])).sum()
print(f"\nEncounters where discharge_disposition_id decodes to 'Expired': {expired_count}")
print("readmitted value counts for those encounters:")
print(df[df['discharge_disposition_id'].astype(int).isin([11, 19, 20, 21])]['readmitted'].value_counts())

Shape: (101766, 50)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-